In [1]:
import torch

In [2]:
from model.transformer import TransformerBlock
from model.cross import XBlock
from model.embedding import StableEmbedding
from model.dag_head import OutputDAG

In [3]:
vocab_size = 10
pad_idx = 0
max_vertices = 12 # decoder
max_seq_len = 6 # encoder

In [4]:
emb_dim = 5

In [5]:
num_heads = 1

In [6]:
factor = 2

In [30]:
target_seq_len = max_vertices // factor

In [7]:
batch_size = 2

In [8]:
encoder_tokens = []
for i in range(batch_size):
    curr_len = torch.randint(1, max_seq_len, (1,)).item()
    encoder_tokens.append(torch.randint(1, vocab_size, (curr_len,)))

In [9]:
encoder_tokens = torch.nested.nested_tensor(encoder_tokens)
encoder_tokens = torch.nested.to_padded_tensor(encoder_tokens, pad_idx, (batch_size, max_seq_len))

C:\Users\John\AppData\Local\Temp\ipykernel_8592\1501289939.py:1: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ..\aten\src\ATen\NestedTensorImpl.cpp:180.)
  encoder_tokens = torch.nested.nested_tensor(encoder_tokens)


In [31]:
target_tokens = []
for i in range(batch_size):
    curr_len = torch.randint(1, target_seq_len, (1,)).item()
    target_tokens.append(torch.randint(1, vocab_size, (curr_len,)))

In [33]:
target_tokens = torch.nested.nested_tensor(target_tokens)
target_tokens = torch.nested.to_padded_tensor(target_tokens, pad_idx, (batch_size, target_seq_len))

In [34]:
target_tokens

tensor([[9, 5, 1, 5, 6, 0],
        [8, 2, 0, 0, 0, 0]])

In [10]:
from utils.data import remove_padding_cols, process_data

In [11]:
encoder_tokens

tensor([[6, 6, 0, 0, 0, 0],
        [5, 0, 0, 0, 0, 0]])

In [12]:
encoder_tokens = remove_padding_cols(encoder_tokens, pad_idx)

In [35]:
target_tokens = remove_padding_cols(target_tokens, pad_idx)

In [13]:
encoder_tokens

tensor([[6, 6],
        [5, 0]])

In [36]:
target_tokens

tensor([[9, 5, 1, 5, 6],
        [8, 2, 0, 0, 0]])

In [37]:
target_lens, _, _, _ = process_data(target_tokens, pad_idx, factor)

In [14]:
token_lens, vertex_lens, token_mask, vertex_mask = process_data(encoder_tokens, pad_idx, factor)

In [15]:
batch_size, encoder_len = encoder_tokens.shape

In [16]:
encoder_len

2

In [17]:
decoder_tokens = torch.arange(0, encoder_len * factor)

In [18]:
decoder_tokens

tensor([0, 1, 2, 3])

In [19]:
decoder_tokens = decoder_tokens.unsqueeze(0).expand(batch_size, -1)

In [20]:
decoder_tokens

tensor([[0, 1, 2, 3],
        [0, 1, 2, 3]])

In [21]:
from torch.nn import Module

In [22]:
from utils.data import self_attn_mask, cross_attn_mask

In [23]:
class CustomModel(Module):
    def __init__(self):
        super().__init__()
        self.vocab_embed = StableEmbedding(vocab_size, emb_dim)
        self.enc_pos_embed = StableEmbedding(max_seq_len, emb_dim)
        self.dec_pos_embed = StableEmbedding(max_vertices, emb_dim)
        self.enc_block_1 = TransformerBlock(emb_dim, num_heads, factor)
        self.dec_block_1 = TransformerBlock(emb_dim, num_heads, factor)
        self.enc_kv_dec_q_xblock = XBlock(emb_dim, num_heads, factor)
        self.dec_kv_enc_q_xblock = XBlock(emb_dim, num_heads, factor)
        self.enc_block_2 = TransformerBlock(emb_dim, num_heads, factor)
        self.dec_block_2 = TransformerBlock(emb_dim, num_heads, factor)
        self.enc_kv_dec_q_xblock2 = XBlock(emb_dim, num_heads, factor)
        self.dec_block_3 = TransformerBlock(emb_dim, num_heads, factor)
        self.output_dag = OutputDAG(emb_dim, vocab_size)

    def forward(self, enc_tokens, dec_tokens, enc_is_pad, dec_is_pad):
        enc_self_attn_mask = self_attn_mask(enc_is_pad)
        # torch scaled_dot_product_attention expects a mask for
        # each head, so we need to unsqueeze the mask
        enc_self_attn_mask = enc_self_attn_mask.unsqueeze(1)
        enc_kv_x_attn_mask = cross_attn_mask(enc_is_pad, dec_is_pad)
        enc_kv_x_attn_mask = enc_kv_x_attn_mask.unsqueeze(1)
        dec_self_attn_mask = self_attn_mask(dec_is_pad)
        dec_self_attn_mask = dec_self_attn_mask.unsqueeze(1)
        dec_kv_x_attn_mask = cross_attn_mask(dec_is_pad, enc_is_pad)
        dec_kv_x_attn_mask = dec_kv_x_attn_mask.unsqueeze(1)
        enc = self.vocab_embed(enc_tokens) + self.enc_pos_embed(torch.arange(0, enc_tokens.shape[1]))

        # dec_tokens are actually the indices of the vertices, so we need only pos embedding
        dec = self.dec_pos_embed(dec_tokens)

        enc = self.enc_block_1(enc, enc_self_attn_mask)
        dec = self.dec_block_1(dec, dec_self_attn_mask)
        enc = self.dec_kv_enc_q_xblock(enc, dec, dec_kv_x_attn_mask)
        dec = self.enc_kv_dec_q_xblock(dec, enc, enc_kv_x_attn_mask)
        enc = self.enc_block_2(enc, enc_self_attn_mask)
        dec = self.dec_block_2(dec, dec_self_attn_mask)
        dec = self.enc_kv_dec_q_xblock2(dec, enc, enc_kv_x_attn_mask)
        dec = self.dec_block_3(dec, dec_self_attn_mask)
        return self.output_dag(dec)

In [24]:
m = CustomModel()

In [25]:
log_transition_probs, vocab_log_probs = m.forward(encoder_tokens, decoder_tokens, token_mask, vertex_mask)

In [39]:
from losses.dag_loss import dag_loss

In [41]:
loss = dag_loss(target_tokens, log_transition_probs, vocab_log_probs, target_lens, vertex_lens)

In [42]:
from torch.optim import Adam

In [43]:
optm = Adam(m.parameters())

In [44]:
optm.zero_grad()

In [45]:
loss.backward()

In [46]:
optm.step()